# 01 — The tokenizer tax on minority languages: English vs MSA vs Masri

**Headline question.** When a tokenizer was trained on a corpus dominated by one
language (call it the *dominant* one), how much more does it charge — in tokens,
which is to say in inference cost, latency, and context budget — to represent the
*same meaning* in a *minority* language?

**Why this matters.** Subword tokenizers are quietly load-bearing infrastructure.
A user whose language is overrepresented in a tokenizer's training data gets
~1 token per word; a user whose language is not gets a fragmented byte-pair
shred at multiple times the rate. Same model, same prompt, same answer — but a
2-3× tax on the minority-language user, every request, forever. This notebook
makes that tax concrete on three tokenizers chosen to bracket the small-model
mech-interp space:

- `EleutherAI/pythia-160m` — English byte-level BPE
- `gpt2` — English byte-level BPE
- `ai-forever/mGPT` — multilingual BPE, ~60 languages incl. Arabic

**The zoom-in.** Within Arabic itself, the same dynamic recurses one level
deeper. Modern Standard Arabic (MSA / Fusha, الفصحى) is the *prestige* register:
the language of news, books, formal writing — heavily represented in any web
corpus. Egyptian Arabic (Masri, مصري) is the *colloquial* register: spoken
daily by ~110M people, but underrepresented in formal corpora. So we ask the
same question one fractal level down: does a tokenizer that *can* see Arabic
charge more for the colloquial register than for the prestige one?

**Pass criteria.** The notebook produces (a) a per-triple table, (b) a visible
tokenisation example, (c) aggregate means by tokenizer × register, (d) a
"tokenizer tax" table normalised against the English baseline, and (e) plots.
The Findings cell at the bottom names what the numbers actually show.

**Inputs.** [`eval/prompts/msa-masri-pairs-v1.json`](../eval/prompts/msa-masri-pairs-v1.json) — 30 hand-crafted minimal triples (English / MSA / Masri).
**Weights:** none. Tokenizers only — this notebook does not load model parameters.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display
from transformers import AutoTokenizer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PAIRS_PATH = REPO / "eval" / "prompts" / "msa-masri-pairs-v1.json"

TOKENIZERS = {
    "pythia-160m": "EleutherAI/pythia-160m",
    "gpt2":        "gpt2",
    "mGPT":        "ai-forever/mGPT",
}

print(f"pairs file: {PAIRS_PATH.relative_to(REPO)}")
print(f"tokenizers: {list(TOKENIZERS)}")

## 2. Load the triple set

Each triple has the same meaning expressed in English, MSA, and Masri. The
notebook treats the *triple* as the unit of comparison; we never compare across
triples.

In [ ]:
pairs = json.loads(PAIRS_PATH.read_text())["pairs"]
print(f"loaded {len(pairs)} triples")
print(f"categories: {sorted({p['category'] for p in pairs})}")

# Show one triple so the rest of the notebook is grounded in something concrete.
display(HTML(
    "<table style='font-size:1.1em'>"
    f"<tr><th>id</th><td>{pairs[8]['id']}</td></tr>"
    f"<tr><th>gloss</th><td>{pairs[8]['gloss_en']}</td></tr>"
    f"<tr><th>English</th><td>{pairs[8]['english']}</td></tr>"
    f"<tr><th>MSA</th><td dir='rtl'>{pairs[8]['msa']}</td></tr>"
    f"<tr><th>Masri</th><td dir='rtl'>{pairs[8]['masri']}</td></tr>"
    "</table>"
))

## 3. Load tokenizers

`AutoTokenizer.from_pretrained` only fetches tokenizer files — no model weights.
First run downloads ~5-20 MB per tokenizer.

In [ ]:
tokenizers: dict[str, "AutoTokenizer"] = {}
for label, hf_name in TOKENIZERS.items():
    tok = AutoTokenizer.from_pretrained(hf_name)
    tokenizers[label] = tok
    print(f"  {label:>11s}  vocab_size = {tok.vocab_size:>6d}  ({hf_name})")

## 4. Per-pair tokenisation table

For every (pair, register, tokenizer) triple, record the token count, character
count, and tokens-per-char (a tokenizer-agnostic efficiency measure).

In [ ]:
def per_pair_rows(pairs: list[dict], tok_label: str, tok) -> list[dict]:
    out = []
    for p in pairs:
        for register in ("english", "msa", "masri"):
            text = p[register]
            ids = tok.encode(text, add_special_tokens=False)
            chars = len(text)
            out.append({
                "id":              p["id"],
                "category":        p["category"],
                "tokenizer":       tok_label,
                "register":        register,
                "text":            text,
                "n_tokens":        len(ids),
                "n_chars":         chars,
                "tokens_per_char": len(ids) / chars if chars else 0.0,
            })
    return out

rows: list[dict] = []
for label, tok in tokenizers.items():
    rows.extend(per_pair_rows(pairs, label, tok))

df = pd.DataFrame(rows)
df.head(6)

## 5. Visible tokenisation — what does each tokenizer actually do?

Numbers are easier to trust when you've seen the pieces. We pick pair `p09`
(*"I want some water"* — `أريد بعض الماء` / `عايز شوية ميه`), encode it under each
tokenizer, decode the IDs back to strings one at a time, and display them
left-to-right (token order in the model's input) regardless of script direction.

In [ ]:
def show_tokenisation(text: str, tok, *, label: str) -> None:
    ids = tok.encode(text, add_special_tokens=False)
    pieces = [tok.decode([i]) for i in ids]
    # Render token boundaries as boxes; the boxes themselves go LTR (token order).
    boxes = "".join(
        f"<span style='border:1px solid #888; padding:1px 4px; margin:1px;"
        f" font-family:monospace; background:#f6f6f6'>{p!r}</span>"
        for p in pieces
    )
    display(HTML(
        f"<div style='margin:6px 0'>"
        f"<b>{label}</b> &mdash; {len(ids)} tokens<br>"
        f"<div dir='ltr' style='line-height:2em'>{boxes}</div>"
        f"</div>"
    ))

example = next(p for p in pairs if p["id"] == "p09")
display(HTML(f"<p>Triple {example['id']}: <b>{example['gloss_en']}</b></p>"))
display(HTML(
    f"<p><b>English:</b> {example['english']}</p>"
    f"<p dir='rtl'><b>MSA:</b> {example['msa']}<br>"
    f"<b>Masri:</b> {example['masri']}</p>"
))

for label, tok in tokenizers.items():
    show_tokenisation(example["english"], tok, label=f"{label}  (English)")
    show_tokenisation(example["msa"],     tok, label=f"{label}  (MSA)")
    show_tokenisation(example["masri"],   tok, label=f"{label}  (Masri)")

## 6. Aggregate — mean tokens per triple, by tokenizer × register

The headline numbers. `delta_masri_minus_msa` is positive when a tokenizer spends
*more* tokens on Masri than on MSA — i.e. when it represents the dialect less
efficiently. `ratio_arabic_over_english` says how much more expensive *Arabic in
either register* is than the same proposition in English on the same tokenizer.

In [ ]:
mean_tokens = (
    df.groupby(["tokenizer", "register"])["n_tokens"]
      .mean()
      .unstack("register")
      [["english", "msa", "masri"]]
      .round(2)
)
mean_tokens["delta_masri_minus_msa"] = (mean_tokens["masri"] - mean_tokens["msa"]).round(2)
mean_tokens["ratio_masri_over_msa"]  = (mean_tokens["masri"] / mean_tokens["msa"]).round(3)
mean_tokens

In [ ]:
mean_bpc = (
    df.groupby(["tokenizer", "register"])["tokens_per_char"]
      .mean()
      .unstack("register")
      [["english", "msa", "masri"]]
      .round(3)
)
mean_bpc["delta_masri_minus_msa"] = (mean_bpc["masri"] - mean_bpc["msa"]).round(3)
mean_bpc

## 7. The tokenizer tax — same meaning, normalised to English

This is the headline. For each tokenizer, we divide its mean Arabic-register
token count by its mean English token count on the *same 30 propositions*. The
result is a unitless multiplier: how many tokens does this tokenizer charge a
Masri (or MSA) speaker for the same meaning, expressed as a multiple of what an
English speaker would pay?

The English column is, by construction, always 1.00× — the reference. A value of
2.00× means "this tokenizer spends twice as many tokens on the Arabic version of
the proposition as on the English one." Read it as inference cost, latency, and
context-window consumption, all of which scale linearly in the token count.

In [ ]:
tax = pd.DataFrame({
    "english_tokens":      mean_tokens["english"],
    "msa_tokens":          mean_tokens["msa"],
    "masri_tokens":        mean_tokens["masri"],
    "english_tax":         (mean_tokens["english"] / mean_tokens["english"]).round(2),
    "msa_tax_x_english":   (mean_tokens["msa"]     / mean_tokens["english"]).round(2),
    "masri_tax_x_english": (mean_tokens["masri"]   / mean_tokens["english"]).round(2),
})
tax

**How to read the tax column.** A multilingual tokenizer with real Arabic
subwords (mGPT) sits at ~1.0× — Arabic costs roughly the same as English on a
per-meaning basis. The English-only BPEs sit at 2.0× (`pythia-160m`) and 2.6×
(`gpt2`): a Masri or MSA user pays double or more for the same answer. This is
not a property of Arabic the language; it is a property of which corpora the
tokenizer's vocabulary was carved out of. The same algorithm trained on
multilingual data produces a far more equitable tokenizer.

## 8. Plots

**Left:** the tokenizer-tax bar chart — bars normalised so English = 1.0 on
each tokenizer; everything above the dashed line is a tax on the minority
language.
**Right:** raw mean tokens per triple, English / MSA / Masri, by tokenizer
(the un-normalised version of the same data, useful as a reality check).

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4.5))
labels = list(mean_tokens.index)
x = range(len(labels))
width = 0.26
ax.bar([i - width for i in x], tax["english_tax"],         width, label="English (reference)", color="#55A868")
ax.bar(list(x),                tax["msa_tax_x_english"],   width, label="MSA",                 color="#4C72B0")
ax.bar([i + width for i in x], tax["masri_tax_x_english"], width, label="Masri",               color="#DD8452")
ax.axhline(1.0, color="grey", linestyle="--", linewidth=1, label="parity with English")
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel("Tokens per English-equivalent meaning  (× English)")
ax.set_title("The tokenizer tax: same meaning, normalised to English")
# annotate each Arabic bar with the multiplier it represents
for i, label in enumerate(labels):
    for offset, col, name in [(0, "msa_tax_x_english", "MSA"), (width, "masri_tax_x_english", "Masri")]:
        v = tax.loc[label, col]
        ax.text(i + offset, v + 0.04, f"{v:.2f}×", ha="center", va="bottom", fontsize=9)
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

## 9. Zoom-in: the prestige-vs-colloquial tax *within* Arabic

The tokenizer-tax above measured the dominant-vs-minority gap *between
languages*: Arabic vs English. The same logic applies to *registers within a
language*. MSA is the prestige register, heavily represented in any web
corpus; Masri is the colloquial register, underrepresented. So we expect a
smaller version of the same effect — but only on the tokenizer that *can* see
Arabic at all (mGPT). On the byte-level English BPEs the dialect signal is
drowned in noise, because they shred all Arabic at the byte level regardless
of register.

**Left:** raw mean tokens per triple, English / MSA / Masri, by tokenizer (the
un-normalised data behind the tax chart, useful as a reality check).
**Right:** per-triple scatter, MSA tokens vs Masri tokens. Points above
`y = x` are triples the tokenizer spends *more* tokens on in Masri than in
MSA — the dialect penalty.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: grouped bars, three registers per tokenizer
ax = axes[0]
labels = list(mean_tokens.index)
x = range(len(labels))
width = 0.26
ax.bar([i - width for i in x], mean_tokens["english"], width, label="English", color="#55A868")
ax.bar(list(x),                mean_tokens["msa"],     width, label="MSA",     color="#4C72B0")
ax.bar([i + width for i in x], mean_tokens["masri"],   width, label="Masri",   color="#DD8452")
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel("Mean tokens per triple")
ax.set_title("Raw mean tokens per triple")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.6)

# --- Right: scatter, MSA tokens vs Masri tokens
ax = axes[1]
colors = {"pythia-160m": "#55A868", "gpt2": "#4C72B0", "mGPT": "#DD8452"}
piv = df.pivot_table(index=["id", "tokenizer"], columns="register", values="n_tokens").reset_index()
for label in tokenizers:
    sub = piv[piv["tokenizer"] == label]
    ax.scatter(sub["msa"], sub["masri"], color=colors[label], label=label, alpha=0.75, s=42)
hi = max(piv["msa"].max(), piv["masri"].max()) + 2
ax.plot([0, hi], [0, hi], color="grey", linestyle="--", linewidth=1, label="parity (MSA = Masri)")
ax.set_xlabel("MSA tokens (per triple)")
ax.set_ylabel("Masri tokens (per triple)")
ax.set_title("Per-triple Masri vs MSA")
ax.legend(fontsize=9, loc="lower right")
ax.grid(linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

## 10. Where is the per-triple Masri-vs-MSA gap largest?

Triples with the largest Masri-minus-MSA token gap, *for mGPT* — the only
tokenizer with a real Arabic vocabulary, hence the only one where the dialect
signal isn't drowned in byte-level noise.

In [ ]:
mgpt = df[df["tokenizer"] == "mGPT"].pivot_table(
    index=["id", "category"], columns="register", values="n_tokens"
).reset_index()
mgpt["delta"] = mgpt["masri"] - mgpt["msa"]
mgpt = mgpt.sort_values("delta", ascending=False)
mgpt.head(10)

## Findings

*(Numbers below are read off the cells above. If you re-run with new triples,
edit this cell to match.)*

1. **The tokenizer tax is real and large: 2-2.6× the tokens for the same
   meaning, on English-only BPEs.** A Masri or MSA speaker using a `gpt2`-
   tokenised model pays **2.59× (MSA) and 2.57× (Masri) more tokens** than
   an English speaker for the same proposition. On `pythia-160m` the
   multiplier is **~1.99× (MSA) and ~1.95× (Masri)**. This is direct
   inflation of inference cost, latency, and context-window consumption —
   one prompt, one answer, but a 2-3× tax that compounds over every
   request the user ever makes. Arabic is not particularly extreme here;
   any language not represented in an English-only tokenizer's training
   corpus would pay a similar tax.

2. **The gap is curatorial, not inherent.** mGPT — same family of
   algorithm (BPE), just trained with multilingual data including Arabic —
   collapses the gap to **0.97× (MSA) and 1.05× (Masri)**. There is no
   technical reason a tokenizer couldn't be equitable across languages;
   the English-only BPEs are not equitable because their training
   distribution wasn't. This is the lever that fixes the tax: include the
   minority language in the vocabulary curation, and the algorithm does
   the rest. Per-character compression (cell 13) tells the same story
   from another angle: mGPT spends ~0.46 tokens/char on Arabic vs ~0.29
   on English; the English-only BPEs spend ~0.85-1.10 on Arabic vs ~0.28
   on English.

3. **The same dynamic recurses one level deeper: MSA vs Masri.** Once we
   have a tokenizer that *can* see Arabic (mGPT), the prestige-vs-
   colloquial gap appears at smaller scale: mGPT spends ~7.4% more tokens
   on Masri than MSA (5.83 vs 5.43 mean tokens per triple, +0.40 delta).
   The English-only BPEs show no such gap because they shred all Arabic
   at the byte level — both registers cost the same to a tokenizer that
   isn't really tokenising either. The same prestige-vs-colloquial bias
   that runs between English and Arabic runs in miniature between MSA and
   Masri, and is visible only on the tokenizer that bothered to learn
   Arabic in the first place.

4. **Per-triple, mGPT's biggest Masri penalties cluster on the dialect-
   marker lexicon and on Masri's aspectual `بـ` prefix.** The top-10
   worst triples (cell 10) all contain one or more of: `عايز`, `دلوقتي`,
   the `بـ`-prefixed imperfect (`بتذاكر`, `بتعمل`, `بتتكلم`), `إمتى`,
   `كوباية`, `ماشي`. These are exactly the items a Masri speaker would
   name as "obviously not Fusha" — the tokenizer agrees, by spending more
   tokens on them.

## Implications for the residual-stream probing experiment (Phase 1 #2)

The tokenizer is upstream of the residual stream. If a tokenizer charges 2×
on English-only BPEs and ~7% extra on the dialect on multilingual ones, the
residual stream inherits that asymmetry: there are *more, finer-grained*
positions per Arabic prompt than per English one, and within Arabic, more per
Masri than per MSA. Any probe of "language" or "dialect" features in early
layers therefore has to disentangle a learned feature from a structural
tokenisation artefact.

**Concrete prediction for the next session:** a linear probe trained on early-
layer residual-stream activations to classify {English, MSA, Masri} should
saturate almost immediately on English-vs-Arabic (different scripts, trivially
separable; the tokenizer-tax effect *is* the signal at that boundary) and then
need much deeper layers to separate MSA from Masri — and even then, the
MSA/Masri boundary will be carried *primarily by the residual-stream positions
of the dialect-marker tokens identified in cell 10*, not by a dedicated
"dialect register" feature. The crucial control is the **length-controlled**
prompt set (Arabic triples pre-trimmed to identical token counts): if MSA-vs-
Masri probe accuracy holds up there, we have evidence for an actual learned
dialect feature; if it collapses to chance, we were reading a tokenisation
artefact all along — the same tokenizer tax of finding #1, surfacing in the
residual stream.

Distinguishing the learned-feature story from the tokenizer-tax story is the
experiment.